# routes/browser

> Route handlers for file browser navigation and file selection

In [ ]:
#| default_exp routes.browser

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Dict, Any, List, Tuple, Callable, Optional
from pathlib import Path

from cjm_fasthtml_file_browser.core.config import FileBrowserConfig, FileBrowserCallbacks
from cjm_fasthtml_file_browser.core.models import BrowserState
from cjm_fasthtml_file_browser.providers.local import LocalFileSystemProvider
from cjm_fasthtml_file_browser.routes.handlers import init_router as init_fb_router, FileBrowserRouters
from cjm_workflow_state.state_store import SQLiteWorkflowStateStore

from cjm_transcription_source_select.models import SourceSelectUrls, SelectedFile
from cjm_transcription_source_select.utils import detect_file_type, is_media_file
from cjm_transcription_source_select.routes.core import (
    get_step_state, update_step_state, get_session_id_from_sess
)
from cjm_transcription_source_select.components.file_browser_panel import (
    get_browser_state, sync_browser_selection,
)
from cjm_transcription_source_select.components.selection_panel import render_selection_panel
from cjm_transcription_source_select.components.stats_panel import render_stats_content

## Folder Media Files Helper

Lists media files (shallow) in a directory for folder bulk-select.

In [ ]:
#| export
def _list_media_in_folder(
    folder_path: str,  # Directory to scan for media files
) -> List[SelectedFile]:  # Media files found as SelectedFile dicts
    """List media files in a directory (shallow, no recursion)."""
    folder = Path(folder_path)
    if not folder.is_dir():
        return []
    results = []
    for child in sorted(folder.iterdir()):
        if child.is_file() and is_media_file(str(child)):
            file_type = detect_file_type(str(child))
            if file_type:
                results.append(SelectedFile(
                    path=str(child),
                    filename=child.name,
                    file_type=file_type,
                    size_bytes=child.stat().st_size,
                    duration=None,
                    format=child.suffix.lstrip("."),
                ))
    return results

## Selection Change Helpers

Logic for toggling files and folders in/out of the selection queue, used by `on_selection_change` callback.

In [ ]:
#| export
def _toggle_file(
    path: str,  # File path to toggle
    selected_files: List[Dict[str, Any]],  # Current selection (mutated)
    selected_folders: List[str],  # Current folder selections (mutated)
    extraction_results: Dict[str, Any],  # Extraction results (mutated)
) -> None:  # Mutates lists in place
    """Toggle a single file in/out of the selection."""
    target = Path(path)
    existing_paths = {f["path"] for f in selected_files}

    if path in existing_paths:
        selected_files[:] = [f for f in selected_files if f["path"] != path]
        extraction_results.pop(path, None)
        # Auto-deselect parent folder if no files from it remain
        parent = str(target.parent)
        if parent in selected_folders:
            remaining = {f["path"] for f in selected_files}
            has_sibling = any(str(Path(p).parent) == parent for p in remaining)
            if not has_sibling:
                selected_folders[:] = [f for f in selected_folders if f != parent]
    else:
        file_type = detect_file_type(path)
        if file_type and target.exists():
            selected_files.append(SelectedFile(
                path=path,
                filename=target.name,
                file_type=file_type,
                size_bytes=target.stat().st_size,
                duration=None,
                format=target.suffix.lstrip("."),
            ))


def _toggle_folder(
    path: str,  # Folder path to toggle
    selected_files: List[Dict[str, Any]],  # Current selection (mutated)
    selected_folders: List[str],  # Current folder selections (mutated)
    extraction_results: Dict[str, Any],  # Extraction results (mutated)
) -> None:  # Mutates lists in place
    """Toggle all media files in a folder (shallow bulk-select)."""
    target = Path(path)
    if path in selected_folders:
        # Remove all files from this folder and deselect it
        folder_file_paths = {str(child) for child in target.iterdir()
                             if child.is_file() and is_media_file(str(child))}
        selected_files[:] = [f for f in selected_files if f["path"] not in folder_file_paths]
        for fp in folder_file_paths:
            extraction_results.pop(fp, None)
        selected_folders[:] = [f for f in selected_folders if f != path]
    else:
        # Add media files not already selected
        all_media = _list_media_in_folder(path)
        if all_media:
            existing_paths = {f["path"] for f in selected_files}
            new_files = [f for f in all_media if f["path"] not in existing_paths]
            selected_files.extend(new_files)
            selected_folders.append(path)

## Router Initialization

Uses the new `init_router` from `cjm-fasthtml-file-browser` which returns `FileBrowserRouters` (browser + collection routers + render callable). Selection logic is wired via `on_selection_change` callback.

In [ ]:
#| export
def init_browser_router(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    provider: LocalFileSystemProvider,  # File system provider
    config: FileBrowserConfig,  # Browser configuration
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # Mutable URL bundle (for OOB rendering)
    home_path: str = "",  # Home directory for nav buttons
    prefix: str = "/browser",  # Route prefix
) -> Tuple[FileBrowserRouters, Callable]:  # (fb_routers, render_panel_fn)
    """Initialize the file browser with selection callbacks."""
    _home = home_path or provider.get_home_path()

    # --- File browser state management ---
    _browser_state = BrowserState(current_path=_home)

    def _fb_state_getter() -> BrowserState:
        """Get browser state with selection synced from step state."""
        return _browser_state

    def _fb_state_setter(state: BrowserState) -> None:
        """Save browser navigation/sort state."""
        _browser_state.current_path = state.current_path
        _browser_state.sort_by = state.sort_by
        _browser_state.sort_descending = state.sort_descending
        _browser_state.selection = state.selection

    # --- Selection change callback (receives request for session access) ---
    def _on_selection_change(selected_paths: List[str], request=None) -> Tuple:
        """Handle selection changes from file browser checkboxes."""
        if request is None:
            return ()

        sess = request.session
        session_id = get_session_id_from_sess(sess)
        step_state = get_step_state(state_store, workflow_id, session_id)
        selected_files = step_state.get("selected_files", [])
        selected_folders = step_state.get("selected_folders", [])
        extraction_results = step_state.get("extraction_results", {})

        # Build old_paths from both files and folders
        old_paths = {f["path"] for f in selected_files}
        old_paths.update(selected_folders)
        new_paths = set(selected_paths)

        # Find added and removed paths
        added = new_paths - old_paths
        removed = old_paths - new_paths

        # Check folder paths
        old_folder_set = set(selected_folders)
        added_folders = {p for p in added if Path(p).is_dir()}
        removed_folders = {p for p in removed if p in old_folder_set}
        added_files = added - added_folders
        removed_files = removed - removed_folders

        # Process removals
        for path in removed_files:
            _toggle_file(path, selected_files, selected_folders, extraction_results)
        for path in removed_folders:
            _toggle_folder(path, selected_files, selected_folders, extraction_results)

        # Process additions
        for path in added_files:
            if path not in {f["path"] for f in selected_files}:
                _toggle_file(path, selected_files, selected_folders, extraction_results)
        for path in added_folders:
            if path not in selected_folders:
                _toggle_folder(path, selected_files, selected_folders, extraction_results)

        # Persist to step state
        update_step_state(
            state_store, workflow_id, session_id,
            selected_files=selected_files,
            selected_folders=selected_folders,
            extraction_results=extraction_results,
            verified=False,
            committed_audio_paths=[],
        )

        # Sync browser selection display
        sync_browser_selection(_browser_state, selected_files, selected_folders)

        # OOB updates for selection and stats panels
        selection_oob = render_selection_panel(selected_files, urls, extraction_results)
        selection_oob.attrs["hx-swap-oob"] = "outerHTML"
        stats_oob = render_stats_content(selected_files, urls, extraction_results, verified=False, oob=True)
        return (selection_oob, stats_oob)

    fb_callbacks = FileBrowserCallbacks(
        on_selection_change=_on_selection_change,
    )

    fb_routers = init_fb_router(
        config=config,
        provider=provider,
        state_getter=_fb_state_getter,
        state_setter=_fb_state_setter,
        route_prefix=prefix,
        callbacks=fb_callbacks,
        home_path=_home,
    )

    # --- Panel render function ---
    def _render_panel(
        selected_files: Optional[List[Dict[str, Any]]] = None,
        selected_folders: Optional[List[str]] = None,
    ) -> Any:
        """Render the complete browser panel, optionally syncing selection state first."""
        if selected_files is not None:
            sync_browser_selection(_browser_state, selected_files, selected_folders or [])
        from cjm_transcription_source_select.components.file_browser_panel import render_browser_panel
        return render_browser_panel(render_fn=fb_routers.render)

    return fb_routers, _render_panel

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()